In [1]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_agentchat.messages import UserMessage

## TODO: Local Model Calling.

model_client = OllamaChatCompletionClient(model="qwen2.5:14b")

In [2]:
input = [UserMessage(content="Assalamualikum, how are you!!", source="user")]
response = await model_client.create(input)
print(response.content)

Waalaikumsalam! How can I assist you today?


# **Memory in `AutoGen`**

In [3]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core.memory import ListMemory, MemoryContent, MemoryMimeType

In [7]:
# Initialize user memory
user_memory = ListMemory()

# Add user preferences to memory
await user_memory.add(MemoryContent(content="The weather should be in metric units", mime_type=MemoryMimeType.TEXT))

await user_memory.add(MemoryContent(content="Meal recipe must be vegan", mime_type=MemoryMimeType.TEXT))

await user_memory.add(MemoryContent(content="User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.", mime_type=MemoryMimeType.TEXT))


async def get_weather(city: str, units: str = "imperial") -> str:
    if units == "imperial":
        return f"The weather in {city} is 73 °F and Sunny."
    elif units == "metric":
        return f"The weather in {city} is 23 °C and Sunny."
    else:
        return f"Sorry, I don't know the weather in {city}."


assistant_agent = AssistantAgent(
    name="assistant_agent",
    model_client=OllamaChatCompletionClient(model="qwen2.5:14b"),
    tools=[get_weather],
    memory=[user_memory],
)


In [8]:
# Run the agent with a task.
stream = assistant_agent.run_stream(task="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.")
await Console(stream)


---------- TextMessage (user) ----------
Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.
---------- MemoryQueryEvent (assistant_agent) ----------
[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None)]
---------- TextMessage (assistant_agent) ----------
I don't have specific information about places to hang out in Bashundhara, Dhaka that match the names "restreamnet" or "spatial place," which might be misspellings. However, I can suggest a few popular spots for hanging out with friends:

1. **Bashundhara Shopping Complex** - A

TaskResult(messages=[TextMessage(id='69eba3ca-b0f9-47fb-950e-289c9e1cd22b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 23, 54, 199679, tzinfo=datetime.timezone.utc), content="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.", type='TextMessage'), MemoryQueryEvent(id='0de09501-8838-4586-8e97-ea3ca0a6418c', source='assistant_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 23, 54, 206032, tzinfo=datetime.timezone.utc), content=[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TE

In [9]:
# Run the agent with a task.
stream = assistant_agent.run_stream(task="What is the weather in Dhaka?")
await Console(stream)


---------- TextMessage (user) ----------
What is the weather in Dhaka?
---------- MemoryQueryEvent (assistant_agent) ----------
[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None)]
---------- ToolCallRequestEvent (assistant_agent) ----------
[FunctionCall(id='0', arguments='{"city": "Dhaka", "units": "metric"}', name='get_weather')]
---------- ToolCallExecutionEvent (assistant_agent) ----------
[FunctionExecutionResult(content='The weather in Dhaka is 23 °C and Sunny.', name='get_weather', call_id='0', is_error=False)]
---------- ToolCallSummaryMessage (assistant_agent) ----------
The weather in Dhaka is 23 °C and Sun

TaskResult(messages=[TextMessage(id='e4374ca3-ade2-4b86-ba43-b9d19faf4a8c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 25, 56, 968409, tzinfo=datetime.timezone.utc), content='What is the weather in Dhaka?', type='TextMessage'), MemoryQueryEvent(id='f3dba6d6-3482-45d3-8634-e75b0a22072c', source='assistant_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 25, 56, 972585, tzinfo=datetime.timezone.utc), content=[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None)], type='MemoryQueryEvent'), ToolCallRequestEvent(id='bc9c16f0-f778-4c63-8ff7-b32

In [19]:
all_messages = await assistant_agent.model_context.get_messages()

all_messages

[UserMessage(content="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.", source='user', type='UserMessage'),
 SystemMessage(content='\nRelevant memory content (in chronological order):\n1. The weather should be in metric units\n2. Meal recipe must be vegan\n3. User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.\n', type='SystemMessage'),
 AssistantMessage(content='I don\'t have specific information about places to hang out in Bashundhara, Dhaka that match the names "restreamnet" or "spatial place," which might be misspellings. However, I can suggest a few popular spots for hanging out with friends:\n\n1. **Bashundhara Shopping Complex** - A large shopping mall with various restaurants and cafes.\n2. **Gulshan Lake Restaurant & Cafe** - Offers an outdoor seating area by the lake for relaxation.\n3. **Mymensingh Road (Road No 9)** - Known for its s

UserMessage(content='What is the weather in Dhaka?', source='user', type='UserMessage')